In [1]:
import os
import re
import random
import math
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from google.colab import drive

In [2]:
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/05/rus.txt'
SAVE_MODEL_PATH = '/content/drive/MyDrive/05/best_seq2seq_attention_model.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Mounted at /content/drive
Using device: cuda


### Data Preprocessing

In [3]:
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
PAD_TOKEN = '<pad>'

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class Vocabulary:
    def __init__(self):
        self.word2index = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2}
        self.index2word = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN}
        self.n_words = 3

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index and len(word) > 0:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

def load_and_preprocess_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')

    eng_vocab = Vocabulary()
    rus_vocab = Vocabulary()
    pairs = []

    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 2:
            eng_sentence = clean_text(parts[0])
            rus_sentence = clean_text(parts[1])

            if eng_sentence and rus_sentence:
                eng_vocab.add_sentence(eng_sentence)
                rus_vocab.add_sentence(rus_sentence)
                pairs.append((eng_sentence, rus_sentence))

    random.seed(42)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * 0.8)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    return eng_vocab, rus_vocab, train_pairs, val_pairs

eng_vocab, rus_vocab, train_pairs, val_pairs = load_and_preprocess_data(DATA_PATH)
print(f"Total train pairs: {len(train_pairs)} | Total val pairs: {len(val_pairs)}")


Total train pairs: 290708 | Total val pairs: 72678


## dataset, data loader

In [4]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, eng_vocab, rus_vocab):
        self.pairs = pairs
        self.eng_vocab = eng_vocab
        self.rus_vocab = rus_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng_sentence, rus_sentence = self.pairs[idx]
        eng_indices = [self.eng_vocab.word2index[w] for w in eng_sentence.split(' ')]
        rus_indices = [self.rus_vocab.word2index[SOS_TOKEN]] + \
                      [self.rus_vocab.word2index[w] for w in rus_sentence.split(' ')] + \
                      [self.rus_vocab.word2index[EOS_TOKEN]]

        return torch.tensor(eng_indices, dtype=torch.long), torch.tensor(rus_indices, dtype=torch.long)

def collate_fn(batch):
    eng_batch, rus_batch = zip(*batch)
    eng_padded = pad_sequence(eng_batch, padding_value=0, batch_first=False)
    rus_padded = pad_sequence(rus_batch, padding_value=0, batch_first=False)
    return eng_padded, rus_padded

BATCH_SIZE = 32
train_dataset = TranslationDataset(train_pairs, eng_vocab, rus_vocab)
val_dataset = TranslationDataset(val_pairs, eng_vocab, rus_vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## seq2seq with attention

In [5]:

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        # return the full sequence of outputs to calculate attention
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # Concatenating hidden state and encoder output
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]

        # Repeat decoder hidden state src_len times
        hidden = hidden.squeeze(0).unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)

        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # LSTM input is now embedding + context vector
        self.lstm = nn.LSTM(hidden_dim + emb_dim, hidden_dim)
        # Output layer takes hidden state, context vector, and embedded input
        self.fc_out = nn.Linear(hidden_dim * 2 + emb_dim, output_dim)

    def forward(self, input_token, hidden, cell, encoder_outputs, mask):
        input_token = input_token.unsqueeze(0)
        embedded = self.embedding(input_token)

        # Calculate attention weights
        a = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)

        # Calculate context vector
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs).permute(1, 0, 2)

        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))

        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)

        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.device = device

    def create_mask(self, src):
        # Mask out <pad> tokens so attention ignores them
        return (src != self.src_pad_idx).permute(1, 0)

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        mask = self.create_mask(src)

        input_token = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell, encoder_outputs, mask)
            outputs[t] = output
            top1 = output.argmax(1)
            teacher_force = random.random() < teacher_forcing_ratio
            input_token = trg[t] if teacher_force else top1

        return outputs

## hyperparams

In [6]:

INPUT_DIM = eng_vocab.n_words
OUTPUT_DIM = rus_vocab.n_words
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 1024
SRC_PAD_IDX = eng_vocab.word2index[PAD_TOKEN]
TRG_PAD_IDX = rus_vocab.word2index[PAD_TOKEN]

model init

In [7]:
attention = Attention(HID_DIM)
encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, attention)
model = Seq2Seq(encoder, decoder, SRC_PAD_IDX, device).to(device)

## training

In [8]:
torch.cuda.empty_cache()

In [9]:
"abc"

'abc'

In [10]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for src, trg in iterator:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in iterator:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0) # Turn off teacher forcing
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

N_EPOCHS = 10
CLIP = 1.0
best_valid_loss = float('inf')

print("Starting Training for Attention Model...")
for epoch in range(N_EPOCHS):
    torch.cuda.empty_cache() # Clear unused CUDA memory before each epoch
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, val_loader, criterion)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), SAVE_MODEL_PATH)
        print(f"Epoch: {epoch+1:02} | Model saved! (Val Loss decreased)")
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f}')

Starting Training for Attention Model...
Epoch: 01 | Model saved! (Val Loss decreased)
Epoch: 01 | Train Loss: 2.787 | Val. Loss: 2.557
Epoch: 02 | Model saved! (Val Loss decreased)
Epoch: 02 | Train Loss: 1.674 | Val. Loss: 2.456
Epoch: 03 | Train Loss: 1.309 | Val. Loss: 2.477
Epoch: 04 | Train Loss: 1.124 | Val. Loss: 2.504


KeyboardInterrupt: 

In [11]:
def translate_greedy(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()
    if isinstance(sentence, str):
        tokens = clean_text(sentence).split(' ')
        indices = [src_vocab.word2index.get(token, 0) for token in tokens]
        tensor = torch.LongTensor(indices).unsqueeze(1).to(device)
    else:
        tensor = sentence.unsqueeze(1).to(device) if sentence.dim() == 1 else sentence.to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(tensor)
        mask = model.create_mask(tensor)

    trg_indexes = [trg_vocab.word2index[SOS_TOKEN]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell, encoder_outputs, mask)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab.word2index[EOS_TOKEN]:
            break

    trg_tokens = [trg_vocab.index2word[i] for i in trg_indexes]
    return trg_tokens[1:-1]

BLEU and beam search

In [12]:
# BLEU Evaluation

def get_ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def clipped_precision(hypothesis, reference, n):
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference, n)
    total_hyp_ngrams = sum(hyp_ngrams.values())
    if total_hyp_ngrams == 0: return 0.0
    clipped_count = sum(min(count, ref_ngrams[ngram]) for ngram, count in hyp_ngrams.items())
    return clipped_count / total_hyp_ngrams

def brevity_penalty(hypothesis, reference):
    c = len(hypothesis)
    r = len(reference)
    if c == 0: return 0.0
    if c > r: return 1.0
    return math.exp(1 - (r / c))

def bleu_score(hypothesis, reference):
    precisions = [clipped_precision(hypothesis, reference, i) for i in range(1, 5)]
    if any(p == 0.0 for p in precisions): return 0.0
    geo_mean = math.exp(sum(math.log(p) for p in precisions) / 4)
    bp = brevity_penalty(hypothesis, reference)
    return bp * geo_mean


# Beam Search

def beam_search(sentence, src_vocab, trg_vocab, model, device, beam_width=3, max_len=50):
    model.eval()
    if isinstance(sentence, str):
        tokens = clean_text(sentence).split(' ')
        indices = [src_vocab.word2index.get(token, 0) for token in tokens]
        tensor = torch.LongTensor(indices).unsqueeze(1).to(device)
    else:
        tensor = sentence.unsqueeze(1).to(device) if sentence.dim() == 1 else sentence.to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(tensor)
        mask = model.create_mask(tensor)

    alive_seqs = [(0.0, [trg_vocab.word2index[SOS_TOKEN]], hidden, cell)]
    completed_seqs = []

    for _ in range(max_len):
        all_candidates = []
        for score, seq, h, c in alive_seqs:
            last_token = seq[-1]
            trg_tensor = torch.LongTensor([last_token]).to(device)
            with torch.no_grad():
                output, new_h, new_c = model.decoder(trg_tensor, h, c, encoder_outputs, mask)

            log_probs = F.log_softmax(output, dim=1).squeeze(0)
            topk_log_probs, topk_indices = log_probs.topk(beam_width)

            for i in range(beam_width):
                cand_score = score + topk_log_probs[i].item()
                cand_seq = seq + [topk_indices[i].item()]
                all_candidates.append((cand_score, cand_seq, new_h, new_c))

        ordered = sorted(all_candidates, key=lambda x: x[0], reverse=True)
        alive_seqs = []

        for cand in ordered:
            if len(alive_seqs) >= beam_width: break
            if cand[1][-1] == trg_vocab.word2index[EOS_TOKEN]:
                completed_seqs.append((cand[0], cand[1]))
            else:
                alive_seqs.append(cand)

        if not alive_seqs: break

    if alive_seqs:
        completed_seqs.extend([(s, seq) for s, seq, _, _ in alive_seqs])

    completed_seqs.sort(key=lambda x: x[0], reverse=True)
    best_seq_indices = completed_seqs[0][1]

    trg_tokens = [trg_vocab.index2word[i] for i in best_seq_indices]
    if trg_tokens[-1] == EOS_TOKEN: return trg_tokens[1:-1]
    return trg_tokens[1:]

comparison

In [13]:
def run_comparison_experiment(val_pairs, eng_vocab, rus_vocab, model, device, num_sentences=100):
    # Load best weights before testing
    model.load_state_dict(torch.load(SAVE_MODEL_PATH, map_location=device))
    model.eval()

    sample_pairs = random.sample(val_pairs, num_sentences)
    configs = {
        'Greedy': {'func': translate_greedy, 'kwargs': {}},
        'Beam (k=3)': {'func': beam_search, 'kwargs': {'beam_width': 3}},
        'Beam (k=5)': {'func': beam_search, 'kwargs': {'beam_width': 5}},
        'Beam (k=10)': {'func': beam_search, 'kwargs': {'beam_width': 10}},
    }

    results = {config: {'bleu': 0.0, 'time': 0.0} for config in configs}
    example_outputs = []

    print(f"\nRunning evaluation on {num_sentences} sentences...\n")

    for name, config in configs.items():
        total_bleu = 0
        start_time = time.time()

        for i, (eng, rus_ref) in enumerate(sample_pairs):
            ref_tokens = rus_ref.split(' ')
            translation = config['func'](eng, eng_vocab, rus_vocab, model, device, **config['kwargs'])
            total_bleu += bleu_score(translation, ref_tokens)

            if name == 'Greedy' and i < 5:
                example_outputs.append({'Source': eng, 'Reference': rus_ref, 'Greedy': ' '.join(translation)})
            elif name == 'Beam (k=5)' and i < 5:
                example_outputs[i]['Beam (k=5)'] = ' '.join(translation)

        end_time = time.time()
        results[name]['bleu'] = (total_bleu / num_sentences) * 100
        results[name]['time'] = (end_time - start_time) / num_sentences

    print("\n" + "="*55)
    print(f"{'Configuration':<15} | {'BLEU-4 Score':<15} | {'Avg Time/Sent (s)':<15}")
    print("-" * 55)
    for name, metrics in results.items():
        print(f"{name:<15} | {metrics['bleu']:<15.4f} | {metrics['time']:<15.4f}")
    print("="*55 + "\n")

    print("5 CONCRETE TRANSLATION EXAMPLES:\n")
    for i, ex in enumerate(example_outputs):
        print(f"Example {i+1}:")
        print(f"  Source:     {ex['Source']}")
        print(f"  Reference:  {ex['Reference']}")
        print(f"  Greedy:     {ex['Greedy']}")
        print(f"  Beam (k=5): {ex['Beam (k=5)']}\n")


In [14]:
run_comparison_experiment(val_pairs, eng_vocab, rus_vocab, model, device, num_sentences=100)


Running evaluation on 100 sentences...


Configuration   | BLEU-4 Score    | Avg Time/Sent (s)
-------------------------------------------------------
Greedy          | 12.5257         | 0.0152         
Beam (k=3)      | 13.1978         | 0.4572         
Beam (k=5)      | 13.0619         | 0.7461         
Beam (k=10)     | 13.0619         | 1.5579         

5 CONCRETE TRANSLATION EXAMPLES:

Example 1:
  Source:     i havent forgotten you
  Reference:  я не забыла тебя
  Greedy:     
  Beam (k=5): 

Example 2:
  Source:     weve done what we could
  Reference:  мы сделали что смогли
  Greedy:     мы сделали то что могли
  Beam (k=5): мы сделали то что могли

Example 3:
  Source:     it was my plan
  Reference:  это был мой план
  Greedy:     мой план
  Beam (k=5): мой план

Example 4:
  Source:     tom noticed that mary was limping
  Reference:  том заметил что мэри хромает
  Greedy:     том заметил что мэри мэри
  Beam (k=5): том заметил что мэри мэри

Example 5:
  Source:     im happ